# Scale to zero, step by step

A hands-on lab. Run one cell at a time. You will see that a Lakebase Autoscaling endpoint
auto-suspends its compute to zero when idle (costing nothing), and wakes on the next query in
about two seconds. There is no stop button to press: idle is free, automatically.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA`
are set and the bank is loaded on your branch.


## Setup, connect


In [ ]:
import os, sys, time, psycopg
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import client, branch_name, PG_PORT, DEFAULT_DB
PROJECT = os.environ['LB_PROJECT']
MINE    = os.environ['LB_BRANCH']
w = client()

def endpoint_state(branch):
    e = list(w.postgres.list_endpoints(parent=branch_name(PROJECT, branch)))[0]
    st = e.status.current_state.value if e.status and e.status.current_state else '?'
    host = e.status.hosts.host if e.status and e.status.hosts else None
    return st, host, (e.endpoint_id or 'primary')
print('connected as', w.current_user.me().user_name)


## Step 1, the endpoint's current state

`ACTIVE` means compute is up and metering. `IDLE` means it auto-suspended to zero: no compute,
no compute cost. A branch you just used is usually ACTIVE; one you have not queried for a few
minutes will be IDLE. This is a control-plane read (the SDK), not SQL.


In [ ]:
st, host, eid = endpoint_state(MINE)
print(f'branch {MINE}: endpoint {eid} is {st}')
print('IDLE = suspended to zero (free); ACTIVE = serving (metering).')


## Step 2, idle scales to zero on its own

You do not stop it. With no queries, the endpoint auto-suspends to `IDLE` and the compute is
released. To watch a cold wake clearly, point at a branch that has been idle (for example
`production` if you have not touched it): change `MINE` to `'production'` in the cell above and
re-run, or just continue on your own branch, the script labels whether it was a cold wake.


## Step 3, wake it on demand with one query, timed

A single `SELECT 1` wakes the endpoint. Waking is about the compute, not any table, so this
works on any branch. The SQL:

```sql
SELECT 1;
```


In [ ]:
st, host, eid = endpoint_state(MINE)
cold = (st == 'IDLE')
print('endpoint is IDLE; timing the wake...' if cold else f'endpoint is {st} (warm, not a cold wake)')
t0 = time.time()
cred = w.postgres.generate_database_credential(endpoint=f'{branch_name(PROJECT, MINE)}/endpoints/{eid}')
conn = psycopg.connect(host=host, port=PG_PORT, dbname=DEFAULT_DB,
                       user=w.current_user.me().user_name, password=cred.token,
                       sslmode='require', autocommit=True, connect_timeout=30)
with conn.cursor() as cur:
    cur.execute('SELECT 1'); cur.fetchone()
dt = (time.time() - t0) * 1000
conn.close()
st2, _, _ = endpoint_state(MINE)
print(f"{'woke IDLE -> ' + st2 if cold else st2}; first query answered in {dt:.0f} ms")


## Step 4, the storage stayed, only compute went away

Suspending compute never touches your data; only the compute is released and then restored.
Confirm the endpoint is serving again.


In [ ]:
st, _, _ = endpoint_state(MINE)
print(f'branch {MINE} endpoint is now {st}')


## What you learned

Idle Lakebase compute costs nothing, automatically, and wakes on demand in about two seconds
(a cold wake measured about 1.7s on a live run; a warm query is faster). You pay only when you
serve. Next: **One copy, both directions** shows operational and analytical work on one copy.
